# Flux de lucru Human-in-the-Loop cu Microsoft Agent Framework

## 🎯 Obiective de învățare

În acest caiet, vei învăța cum să implementezi fluxuri de lucru **human-in-the-loop** folosind `RequestInfoExecutor` din Microsoft Agent Framework. Acest model puternic îți permite să pui în pauză fluxurile de lucru AI pentru a aduna input uman, făcând agenții tăi interactivi și oferindu-le oamenilor controlul asupra deciziilor critice.

## 🔄 Ce este Human-in-the-Loop?

**Human-in-the-loop (HITL)** este un model de design în care agenții AI pun în pauză execuția pentru a cere input uman înainte de a continua. Acest lucru este esențial pentru:

- ✅ **Decizii critice** - Obține aprobarea umană înainte de a lua acțiuni importante
- ✅ **Situații ambigue** - Permite oamenilor să clarifice când AI are incertitudini
- ✅ **Preferințe ale utilizatorilor** - Cere utilizatorilor să aleagă între multiple opțiuni
- ✅ **Conformitate și siguranță** - Asigură supraveghere umană pentru operațiuni reglementate
- ✅ **Experiențe interactive** - Construiește agenți conversaționali care răspund la inputul utilizatorului

## 🏗️ Cum funcționează în Microsoft Agent Framework

Framework-ul oferă trei componente cheie pentru HITL:

1. **`RequestInfoExecutor`** - Un executor special care pune în pauză fluxul de lucru și emite un `RequestInfoEvent`
2. **`RequestInfoMessage`** - Clasa de bază pentru payload-uri de cerere tipizate trimise oamenilor
3. **`RequestResponse`** - Corelează răspunsurile umane cu cererile originale folosind `request_id`

**Model flux de lucru:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Exemplul nostru: Rezervare hotel cu confirmare utilizator

Vom construi pe baza fluxului de lucru condițional adăugând confirmarea umană **înainte** de a sugera destinații alternative:

1. Utilizatorul solicită o destinație (ex: "Paris")
2. `availability_agent` verifică dacă sunt camere disponibile
3. **Dacă nu sunt camere** → `confirmation_agent` întreabă „Doriți să vedeți alternative?”
4. Fluxul de lucru **se oprește** folosind `RequestInfoExecutor`
5. **Omul răspunde** „da” sau „nu” prin input în consolă
6. `decision_manager` direcționează în funcție de răspuns:
   - **Da** → Afișează destinații alternative
   - **Nu** → Anulează solicitarea de rezervare
7. Afișează rezultatul final

Acest exemplu demonstrează cum să oferi utilizatorilor control asupra sugestiilor agentului!

---

Să începem! 🚀


## Pasul 1: Importarea bibliotecilor necesare

Importăm componentele standard ale Agent Framework plus **clase specifice pentru interacțiunea om-mașină**:
- `RequestInfoExecutor` - Executor care întrerupe fluxul de lucru pentru input uman
- `RequestInfoEvent` - Eveniment emis atunci când se solicită input uman
- `RequestInfoMessage` - Clasă de bază pentru payload-uri tipizate de solicitare
- `RequestResponse` - Corelează răspunsurile umane cu solicitările
- `WorkflowOutputEvent` - Eveniment pentru detectarea ieșirilor din fluxul de lucru


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Pasul 2: Definește Modele Pydantic pentru Ieșiri Structurate

Aceste modele definesc **schema** pe care agenții o vor returna. Păstrăm toate modelele din fluxul de lucru condițional și adăugăm:

**Nou pentru intervenția umană:**
- `HumanFeedbackRequest` - Subclasă a `RequestInfoMessage` care definește payload-ul cererii trimise către oameni
  - Conține `prompt` (întrebarea de adresat) și `destination` (context despre orașul indisponibil)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Pasul 3: Creează instrumentul de rezervare a hotelului

Același instrument din fluxul de lucru condițional - verifică dacă sunt disponibile camere în destinație.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Pasul 4: Definirea Funcțiilor de Condiție pentru Rutare

Avem nevoie de **patru funcții de condiție** pentru fluxul nostru de lucru cu intervenție umană:

**Din fluxul condiționat existent:**
1. `has_availability_condition` - Routează când hotelurile SUNT disponibile
2. `no_availability_condition` - Routează când hotelurile NU sunt disponibile

**Nou pentru intervenția umană:**
3. `user_wants_alternatives_condition` - Routează când utilizatorul spune "da" la alternative
4. `user_declines_alternatives_condition` - Routează când utilizatorul spune "nu" la alternative


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Pasul 5: Creează Executorul Decision Manager

Acesta este **nucleul modelului human-in-the-loop**! `DecisionManager` este un `Executor` personalizat care:

1. **Primește feedback uman** prin obiectele `RequestResponse`
2. **Procesează decizia utilizatorului** (da/nu)
3. **Dirijează fluxul de lucru** trimițând mesaje agenților potriviți

Caracteristici cheie:
- Folosește decoratori `@handler` pentru a expune metode ca pași de lucru
- Primește `RequestResponse[HumanFeedbackRequest, str]` care conține atât cererea originală, cât și răspunsul utilizatorului
- Emite mesaje simple "da" sau "nu" care declanșează funcțiile noastre de condiție


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Pasul 6: Creați Executorul Personalizat de Afișare

Același executor de afișare din fluxul de lucru condițional - returnează rezultatele finale ca ieșire a fluxului de lucru.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Pasul 7: Încarcă variabilele de mediu

Configurează clientul LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Pasul 8: Crearea agenților AI și a executorilor

Creăm **șase componente de flux de lucru**:

**Agenți (încapsulați în AgentExecutor):**
1. **availability_agent** - Verifică disponibilitatea hotelului utilizând instrumentul
2. **confirmation_agent** - 🆕 Pregătește cererea de confirmare umană
3. **alternative_agent** - Sugerează orașe alternative (când utilizatorul spune da)
4. **booking_agent** - Încurajează rezervarea (când sunt camere disponibile)
5. **cancellation_agent** - 🆕 Gestionează mesajul de anulare (când utilizatorul spune nu)

**Executori speciali:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` care oprește fluxul de lucru pentru input uman
7. **decision_manager** - 🆕 Executor personalizat care direcționează pe baza răspunsului uman (deja definit mai sus)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Pasul 9: Construirea fluxului de lucru cu intervenție umană în buclă

Acum construim graficul fluxului de lucru cu **rutare condiționată** incluzând calea cu intervenție umană în buclă:

**Structura fluxului de lucru:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Muchii cheie:**
- `availability_agent → confirmation_agent` (când nu sunt camere)
- `confirmation_agent → prepare_human_request` (transformă tipul)
- `prepare_human_request → request_info_executor` (pauză pentru om)
- `request_info_executor → decision_manager` (întotdeauna - furnizează RequestResponse)
- `decision_manager → alternative_agent` (când utilizatorul spune "da")
- `decision_manager → cancellation_agent` (când utilizatorul spune "nu")
- `availability_agent → booking_agent` (când sunt camere disponibile)
- Toate căile se termină la `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Pasul 10: Rulați Cazul de Test 1 - Oraș FĂRĂ Disponibilitate (Paris cu Confirmare Umană)

Acest test demonstrează **ciclul complet cu intervenție umană**:

1. Cerere hotel în Paris
2. availability_agent verifică → Nu sunt camere
3. confirmation_agent creează o întrebare pentru utilizator
4. request_info_executor **oprește fluxul de lucru** și emite `RequestInfoEvent`
5. **Aplicația detectează evenimentul și solicită utilizatorului în consolă**
6. Utilizatorul tastează "da" sau "nu"
7. Aplicația trimite răspunsul prin `send_responses_streaming()`
8. decision_manager direcționează pe baza răspunsului
9. Rezultatul final este afișat

**Model Cheie:**
- Folosiți `workflow.run_stream()` pentru prima iterație
- Folosiți `workflow.send_responses_streaming(pending_responses)` pentru iterațiile următoare
- Ascultați `RequestInfoEvent` pentru a detecta când este nevoie de input uman
- Ascultați `WorkflowOutputEvent` pentru a captura rezultatele finale


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Pasul 11: Rulează Cazul de Test 2 - Oraș CU Disponibilitate (Stockholm - Fără Intrare Umană Necesara)

Acest test demonstrează **calea directă** când camerele sunt disponibile:

1. Solicită hotel în Stockholm
2. availability_agent verifică → Camere disponibile ✅
3. booking_agent sugerează rezervarea
4. display_result afișează confirmarea
5. **Nu este necesară nicio intervenție umană!**

Fluxul de lucru ocolește complet calea cu intervenție umană când camerele sunt disponibile.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Puncte Cheie și Cele Mai Bune Practici pentru Intervenția Umană în Buclă

### ✅ Ce Ai Învățat:

#### 1. **Modelul RequestInfoExecutor**
Modelul uman în buclă din Microsoft Agent Framework folosește trei componente cheie:
- `RequestInfoExecutor` - Pune fluxul de lucru pe pauză și emite evenimente
- `RequestInfoMessage` - Clasă de bază pentru payload-uri de cereri tipizate (subclasați aceasta!)
- `RequestResponse` - Corelează răspunsurile umane cu cererile originale

**Înțelegere Critică:**
- `RequestInfoExecutor` NU colectează inputul propriu-zis - doar pune fluxul de lucru pe pauză
- Codul aplicației tale trebuie să asculte pentru `RequestInfoEvent` și să colecteze inputul
- Trebuie să apelezi `send_responses_streaming()` cu un dicționar ce mapă `request_id` la răspunsul utilizatorului

#### 2. **Modelul de Execuție în Streaming**
```python
# Prima iterație
stream = workflow.run_stream(initial_request)

# Iterații ulterioare (după input-ul uman)
stream = workflow.send_responses_streaming(pending_responses)

# Procesează întotdeauna evenimentele
events = [event async for event in stream]
```

#### 3. **Arhitectură Bazată pe Evenimente**
Ascultă evenimente specifice pentru a controla fluxul de lucru:
- `RequestInfoEvent` - Este necesar input uman (fluxul de lucru pus pe pauză)
- `WorkflowOutputEvent` - Rezultatul final este disponibil (fluxul de lucru complet)
- `WorkflowStatusEvent` - Schimbări de stare (ÎN PROGRES, INACTIV CU CERERI ÎN AȘTEPTARE, etc.)

#### 4. **Executorii Personalizați cu @handler**
`DecisionManager` demonstrează cum să creezi executori care:
- Folosesc decoratorul `@handler` pentru a expune metode ca pași ai fluxului de lucru
- Primesc mesaje tipizate (de ex., `RequestResponse[HumanFeedbackRequest, str]`)
- Direcționează fluxul de lucru trimițând mesaje altor executori
- Accesează contextul prin `WorkflowContext`

#### 5. **Rutare Condiționată cu Decizii Umane**
Poți crea funcții condiționale care evaluează răspunsurile umane:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Aplicații în Viața Reală:

1. **Fluxuri de Aprobare**
   - Obține aprobarea managerului înainte de procesarea rapoartelor de cheltuieli
   - Necesită revizuire umană înainte de trimiterea emailurilor automate
   - Confirmă tranzacțiile de valoare mare înainte de execuție

2. **Moderarea Conținutului**
   - Marchează conținutul discutabil pentru revizuire umană
   - Cere moderatorilor să ia decizia finală în cazuri marginale
   - Escalează către oameni când încrederea AI este scăzută

3. **Serviciul Clienți**
   - Lasă AI să gestioneze automat întrebările de rutină
   - Escalează problemele complexe către agenți umani
   - Întreabă clientul dacă dorește să vorbească cu un om

4. **Procesarea Datelor**
   - Cere oamenilor să rezolve intrările de date ambigue
   - Confirmă interpretările AI ale documentelor neclare
   - Permite utilizatorilor să aleagă între multiple interpretări valide

5. **Sisteme Critice pentru Siguranță**
   - Necesită confirmare umană înainte de acțiuni ireversibile
   - Obține aprobare înainte de accesarea datelor sensibile
   - Confirmă deciziile în industrii reglementate (sănătate, finanțe)

6. **Agenți Interactivi**
   - Construiește roboți conversaționali care pun întrebări suplimentare
   - Creează vrăjitori care ghidează utilizatorii prin procese complexe
   - Proiectează agenți care colaborează cu oamenii pas cu pas

### 🔄 Comparație: Cu vs Fără Uman în Buclă

| Funcționalitate | Flux Condiționat | Flux cu Uman în Buclă |
|---------|---------------------|---------------------------|
| **Execuție** | Un singur `workflow.run()` | Buclă cu `run_stream()` + `send_responses_streaming()` |
| **Input Utilizator** | Niciunul (complet automatizat) | Prompturi interactive prin `input()` sau UI |
| **Componente** | Agenți + Executori | + RequestInfoExecutor + DecisionManager |
| **Evenimente** | Doar AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, etc. |
| **Pauză** | Fără pauză | Fluxul de lucru se oprește la RequestInfoExecutor |
| **Control Uman** | Fără control uman | Oamenii iau decizii cheie |
| **Caz de Utilizare** | Automatizare | Colaborare și supraveghere |

### 🚀 Modele Avansate:

#### Puncte Multiple de Decizie Umană
Poți avea noduri multiple `RequestInfoExecutor` în același flux de lucru:
```python
.add_edge(agent1, request_info_1)  # Prima decizie umană
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # A doua decizie umană
.add_edge(decision_manager_2, final_agent)
```

#### Gestionarea Timeout-ului
Implementează timeout-uri pentru răspunsurile umane:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Implicit la opțiunea sigură
```

#### Integrare UI Complexă
În loc de `input()`, integrează cu UI web, Slack, Teams, etc.:
```python
if isinstance(event, RequestInfoEvent):
    # Trimite notificare pe canalul preferat al utilizatorului
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Human-in-the-Loop Condiționat
Cere input uman numai în situații specifice:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Trimiteți doar către om dacă încrederea este scăzută sau valoarea este ridicată
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Cele Mai Bune Practici:

1. **Subclass întotdeauna RequestInfoMessage**
   - Oferă siguranță de tip și validare
   - Permite context bogat pentru redarea UI
   - Clarifică intenția fiecărui tip de cerere

2. **Folosește Prompturi Descriptive**
   - Include context despre ce întrebi
   - Explică consecințele fiecărei alegeri
   - Păstrează întrebările simple și clare

3. **Gestionează Inputul Neprevăzut**
   - Validatează răspunsurile utilizatorilor
   - Oferă valori implicite pentru input invalid
   - Furnizează mesaje clare de eroare

4. **Urmărește ID-urile Cererilor**
   - Folosește corelația dintre request_id și răspunsuri
   - Nu încerca să gestionezi starea manual

5. **Proiectează pentru Non-Blocare**
   - Nu bloca thread-urile în așteptarea inputului
   - Folosește modele asincrone peste tot
   - Suportă instanțe concurente ale fluxului de lucru

### 📚 Concepte Relaționate:

- **Middleware pentru Agenți** - Interceptează apelurile agenților (notebook anterior)
- **Managementul Stării Fluxului de Lucru** - Persistă starea fluxului între rulări
- **Colaborare Multi-Agenți** - Combină umanul în buclă cu echipe de agenți
- **Arhitecturi Bazate pe Evenimente** - Construiește sisteme reactive cu evenimente

---

### 🎓 Felicitări!

Ai stăpânit fluxurile de lucru uman în buclă cu Microsoft Agent Framework! Acum știi cum să:
- ✅ Pui fluxurile pe pauză pentru a aduna input uman
- ✅ Folosești RequestInfoExecutor și RequestInfoMessage
- ✅ Gestionezi execuția în streaming cu evenimente
- ✅ Creezi executori personalizați cu @handler
- ✅ Rutezi fluxurile pe baza deciziilor umane
- ✅ Construiești agenți AI interactivi care colaborează cu oamenii

**Acesta este un model critic pentru construirea sistemelor AI de încredere și controlabile!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
